In [1]:
!pip install torch datasets transformers tiktoken

In [3]:
from datasets import load_dataset

ds = load_dataset("disham993/alpaca-train-validation-test-split")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-56663e4dc2b054(…):   0%|          | 0.00/17.1M [00:00<?, ?B/s]

data/validation-00000-of-00001-f04e3b4e4(…):   0%|          | 0.00/3.69M [00:00<?, ?B/s]

data/test-00000-of-00001-9f6a652027f1925(…):   0%|          | 0.00/3.69M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36401 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7801 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7800 [00:00<?, ? examples/s]

In [4]:
print(f"Train: {len(ds['train'])}")
print(f"Validation: {len(ds['validation'])}")
print(f"Test: {len(ds['test'])}")

Train: 36401
Validation: 7801
Test: 7800


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-70m")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [6]:
def format_prompt(example):
      instruction = example['instruction']
      output = example['output']


      text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
      return {'text': text}

train_formatted = ds['train'].map(format_prompt, remove_columns=ds['train'].column_names)
test_formatted = ds['test'].map(format_prompt, remove_columns=ds['test'].column_names)

Map:   0%|          | 0/36401 [00:00<?, ? examples/s]

Map:   0%|          | 0/7800 [00:00<?, ? examples/s]

In [7]:
#Custom Byte-Level Tokenizer
class SimpleTokenizer:
    """Simple tokenizer for vocab_size=256 (byte-level)"""
    def __init__(self):
        self.vocab_size = 256
        self.pad_token_id = 2
        self.eos_token_id = 3
        self.pad_token = '<pad>'
        self.eos_token = '<eos>'

    def encode(self, text, truncation=True, max_length=512, padding='max_length'):
        ids = [b for b in text.encode('utf-8')[:max_length - 1]]
        ids.append(self.eos_token_id)
        if padding == 'max_length':
            ids = ids[:max_length] + [self.pad_token_id] * (max_length - len(ids))
        attn = [1 if i != self.pad_token_id else 0 for i in ids]
        return {'input_ids': ids, 'attention_mask': attn}

    def __call__(self, texts, truncation=True, max_length=512,
                 padding='max_length', return_tensors=None):
        if isinstance(texts, str):
            result = self.encode(texts, truncation, max_length, padding)
            if return_tensors == 'pt':
                import torch
                return {k: torch.tensor([v]) for k, v in result.items()}
            return result

        results = [self.encode(t, truncation, max_length, padding) for t in texts]
        batched = {
            'input_ids': [r['input_ids'] for r in results],
            'attention_mask': [r['attention_mask'] for r in results],
        }
        if return_tensors == 'pt':
            import torch
            return {k: torch.tensor(v) for k, v in batched.items()}
        return batched

    def decode(self, ids, skip_special_tokens=True):
        if skip_special_tokens:
            ids = [i for i in ids if i not in [self.pad_token_id, self.eos_token_id]]
        return bytes(ids).decode('utf-8', errors='ignore')

    def save_pretrained(self, path):
        import json, os
        os.makedirs(path, exist_ok=True)
        with open(f'{path}/tokenizer_config.json', 'w') as f:
            json.dump({'vocab_size': self.vocab_size,
                       'pad_token_id': self.pad_token_id,
                       'eos_token_id': self.eos_token_id}, f)

tokenizer = SimpleTokenizer()

In [8]:
#format and tokenize dataset
max_length = 105

def format_prompt(example):
    instruction = example['instruction']
    output = example['output']
    text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
    return {'text': text}

train_formatted = ds['train'].map(format_prompt, remove_columns=ds['train'].column_names)
test_formatted  = ds['test'].map(format_prompt,  remove_columns=ds['test'].column_names)

def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        max_length=max_length,
        padding='max_length',
    )

train_tokenized = train_formatted.map(tokenize, remove_columns=['text'])
test_tokenized  = test_formatted.map(tokenize,  remove_columns=['text'])

train_tokenized = train_tokenized.map(lambda x: {'labels': x['input_ids']})
test_tokenized  = test_tokenized.map(lambda x:  {'labels': x['input_ids']})

Map:   0%|          | 0/36401 [00:00<?, ? examples/s]

Map:   0%|          | 0/7800 [00:00<?, ? examples/s]

Map:   0%|          | 0/36401 [00:00<?, ? examples/s]

Map:   0%|          | 0/7800 [00:00<?, ? examples/s]

Map:   0%|          | 0/36401 [00:00<?, ? examples/s]

Map:   0%|          | 0/7800 [00:00<?, ? examples/s]

In [9]:
#embedding and model architecture
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

class RotaryEmbedding(nn.Module):
    """Rotary Position Embeddings (RoPE)."""
    def __init__(self, head_dim):
        super().__init__()
        # inv_freq shape: [head_dim // 2]
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, seq_len, device):
        # t: [seq_len]
        t = torch.arange(seq_len, device=device).float()
        # freqs: [seq_len, head_dim // 2]
        freqs = torch.outer(t, self.inv_freq)
        # emb: [seq_len, head_dim]
        emb = torch.cat((freqs, freqs), dim=-1)
        # broadcast shape: [1, 1, seq_len, head_dim]
        return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]


def rotate_half(x):
    """Rotate second half of head_dim to first half (pure PyTorch)."""
    half = x.shape[-1] // 2
    return torch.cat((-x[..., half:], x[..., :half]), dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    """Apply RoPE to q and k. Both are [B, H, S, D]."""
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


In [10]:
#attention mechanism with RoPE
class Attention(nn.Module):
    """Multi-head Causal Self-Attention with RoPE."""
    def __init__(self, dim, num_heads, max_seq_len):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5

        self.qkv        = nn.Linear(dim, dim * 3, bias=False)
        self.out        = nn.Linear(dim, dim, bias=False)
        self.rotary_emb = RotaryEmbedding(self.head_dim)

    def forward(self, x, mask=None):
        B, S, D = x.shape

        # Project and split into Q, K, V
        qkv = self.qkv(x)                          # [B, S, 3*D]
        q, k, v = qkv.chunk(3, dim=-1)             # each [B, S, D]

        # Reshape to [B, H, S, head_dim]
        def split_heads(t):
            return t.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        # Apply RoPE — cos/sin are [1, 1, S, head_dim]
        cos, sin = self.rotary_emb(S, x.device)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) * self.scale   # [B, H, S, S]

        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        out  = attn @ v                             # [B, H, S, head_dim]

        # Merge heads [B, S, D]
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.out(out)


In [11]:
# ── FeedForward (SwiGLU) ──────────────────────────────────────────────────────
class FeedForward(nn.Module):
    """SwiGLU Feed-Forward Network."""
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.gate = nn.Linear(dim, hidden_dim, bias=False)
        self.fc   = nn.Linear(dim, hidden_dim, bias=False)
        self.out  = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.out(F.silu(self.gate(x)) * self.fc(x))


# ── Transformer Block ─────────────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    """Single Transformer Block with Pre-Norm."""
    def __init__(self, dim, num_heads, hidden_dim, max_seq_len):
        super().__init__()
        self.attn_norm = nn.LayerNorm(dim)
        self.attn      = Attention(dim, num_heads, max_seq_len)
        self.ffn_norm  = nn.LayerNorm(dim)
        self.ffn       = FeedForward(dim, hidden_dim)

    def forward(self, x, mask=None):
        x = x + self.attn(self.attn_norm(x), mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x

In [12]:
class SimpleSLM(nn.Module):
    """Simple Small Language Model — ~19.8M parameters."""
    def __init__(self, vocab_size=256, dim=396, num_heads=6, num_layers=8,
                 hidden_dim=1536, max_seq_len=105):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb   = nn.Embedding(vocab_size, dim)
        self.layers      = nn.ModuleList([
            TransformerBlock(dim, num_heads, hidden_dim, max_seq_len)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.token_emb.weight = self.head.weight  # weight tying

    def forward(self, x, mask=None, labels=None):
        S = x.shape[1]
        if mask is None:
            mask = torch.tril(torch.ones(S, S, device=x.device))
        x      = self.token_emb(x)
        for layer in self.layers:
            x  = layer(x, mask)
        x      = self.norm(x)
        logits = self.head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )
        return {'logits': logits, 'loss': loss} if loss is not None else logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=100, temperature=0.7, eos_token_id=3):
        generated = []
        for _ in range(max_new_tokens):
            logits   = self(input_ids[:, -self.max_seq_len:])[:, -1, :] / max(temperature, 1e-8)
            probs    = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            generated.append(next_tok.item())
            input_ids = torch.cat([input_ids, next_tok], dim=1)
            if next_tok.item() == eos_token_id:
                break
        return generated

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SimpleSLM(
    vocab_size=256, dim=396, num_heads=6,
    num_layers=8, hidden_dim=1536, max_seq_len=105,
).to(device)

total = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total:,} ({total/1e6:.2f}M)")



Model parameters: 19,731,096 (19.73M)


In [14]:
# ── Cell 6: DataLoader ────────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader


class InstructionDataset(Dataset):
    def __init__(self, data):
        self.input_ids = data['input_ids']
        self.labels    = data['labels']

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids[idx], dtype=torch.long),
            'labels':    torch.tensor(self.labels[idx],    dtype=torch.long),
        }


batch_size   = 4
train_loader = DataLoader(InstructionDataset(train_tokenized), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(InstructionDataset(test_tokenized),  batch_size=batch_size)

In [15]:
# ── Cell 7: Optimizer & Scheduler ────────────────────────────────────────────
from transformers import get_linear_schedule_with_warmup

num_epochs    = 3
learning_rate = 2e-4

optimizer   = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
total_steps = len(train_loader) * num_epochs
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps,
)

In [16]:
#training and evalution functions
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        labels    = batch['labels'].to(device)

        optimizer.zero_grad()
        output = model(input_ids, labels=labels)
        loss   = output['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            labels    = batch['labels'].to(device)
            output    = model(input_ids, labels=labels)
            total_loss += output['loss'].item()
    return total_loss / len(loader)



In [17]:
import torch

if torch.cuda.is_available():
    print("GPU is available")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available")

GPU is available
GPU Name: Tesla T4


In [18]:
#training loop
print(f"Training on {device}...")
print(f"Batches per epoch: {len(train_loader)}")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    test_loss  = evaluate(model, test_loader, device)
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Test  Loss: {test_loss:.4f}")

print("\nTraining complete!")

Training on cuda...
Batches per epoch: 9101

Epoch 1/3
  Train Loss: 0.9962
  Test  Loss: 0.7127

Epoch 2/3
  Train Loss: 0.6350
  Test  Loss: 0.6072

Epoch 3/3
  Train Loss: 0.5100
  Test  Loss: 0.5670

Training complete!


In [28]:
import os

SAVE_DIR = "nano_instruct_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model weights
torch.save(model.state_dict(), f"{SAVE_DIR}/model.pt")

# Save config
import json
config = {
    "vocab_size": 256, "dim": 396, "num_heads": 6,
    "num_layers": 8, "hidden_dim": 1536, "max_seq_len": 105,
}
with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump(config, f, indent=2)

# Save tokenizer
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to '{SAVE_DIR}/'")
print(f"  - {SAVE_DIR}/model.pt")
print(f"  - {SAVE_DIR}/config.json")
print(f"  - {SAVE_DIR}/tokenizer_config.json")

Model saved to 'nano_instruct_model/'
  - nano_instruct_model/model.pt
  - nano_instruct_model/config.json
  - nano_instruct_model/tokenizer_config.json


In [31]:
def generate_response(instruction, max_new_tokens=100, temperature=0.7):
    model.eval()
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    # Encode WITHOUT padding — only real tokens, no pad noise
    prompt_bytes = list(prompt.encode("utf-8")[: max_length - 1])
    input_ids    = torch.tensor([prompt_bytes], dtype=torch.long).to(device)
    prompt_len   = input_ids.shape[1]

    # Generate new token ids
    new_token_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        eos_token_id=tokenizer.eos_token_id,
    )

    if not new_token_ids:
        return "[No output generated]"

    # Decode only the newly generated bytes
    response = tokenizer.decode(new_token_ids, skip_special_tokens=True)
    return response.strip() or "[Empty response — model needs more training]"


# Example
print("=" * 50)
response = generate_response("explain AI in one senetnce")
print("Response:", response)
print("=" * 50)

Response: AI is a computer assistant that uses decision
